In [1]:
# Instala dependências e conecta o Google Drive
# Instala as bibliotecas mais recentes do Hugging Face
!pip install -q transformers datasets accelerate tokenizers

from google.colab import drive
import os

# Monta o Drive para salvar arquivos de forma segura
drive.mount('/content/drive')

# Cria o diretório do projeto no Drive se ele não existir
PASTA_PROJETO = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
os.makedirs(PASTA_PROJETO, exist_ok=True)
print(f"Diretório de trabalho pronto em: {PASTA_PROJETO}")

Mounted at /content/drive
Diretório de trabalho pronto em: /content/drive/MyDrive/Colab_LLMs/Maia_Lite


In [3]:
# Install dependencies and connect Google Drive
# Install the latest Hugging Face libraries
!pip install -q transformers datasets accelerate tokenizers

from google.colab import drive
import os

# Mount Drive to save files securely
drive.mount('/content/drive')

# Create the project directory in Drive if it does not exist
PASTA_PROJETO = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
os.makedirs(PASTA_PROJETO, exist_ok=True)
print(f"Diretório de trabalho pronto em: {PASTA_PROJETO}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Diretório de trabalho pronto em: /content/drive/MyDrive/Colab_LLMs/Maia_Lite


## Configuração do Token do Hugging Face Hub

Para evitar avisos de requisições não autenticadas e potencialmente acelerar os downloads de modelos e datasets, é recomendável configurar um token de acesso do Hugging Face Hub. Siga os passos abaixo:

1.  **Obtenha seu Token de Acesso:**
    *   Vá para [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
    *   Crie um novo token. Recomenda-se um token com permissão `read` (leitura) para a maioria das operações de download.

2.  **Adicione o Token aos Segredos do Colab:**
    *   No painel esquerdo do Google Colab, clique no ícone de chave (🔑) para abrir a interface de "Segredos".
    *   Clique em "Adicionar novo segredo".
    *   No campo "Nome", digite `HF_TOKEN`.
    *   No campo "Valor", cole o token que você obteve do Hugging Face.
    *   Certifique-se de ativar a opção "Acesso ao notebook" para que o notebook possa usar este segredo.

3.  **Execute a célula Python abaixo:**
    *   Esta célula carregará o token dos segredos do Colab e o configurará como uma variável de ambiente, que será usada pelas bibliotecas do Hugging Face.

In [4]:
# Importe as bibliotecas necessárias
from google.colab import userdata
import os
from huggingface_hub import login # Importa a função login do Hugging Face Hub

# Carregue o token do Hugging Face dos segredos do Colab
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        # Tenta fazer o login com o token do Hugging Face
        login(token=hf_token, add_to_git_credential=False) # 'add_to_git_credential=False' para não solicitar credenciais git
        print("Token do Hugging Face carregado e configurado com sucesso via huggingface_hub.login().")
    else:
        print("AVISO: O segredo 'HF_TOKEN' foi encontrado, mas está vazio ou é None. Por favor, verifique o valor nos segredos do Colab.")
        # Se o token estiver vazio/None, ainda define a variável de ambiente como string vazia para evitar erros posteriores
        os.environ['HF_TOKEN'] = ''
except userdata.SecretNotFoundError:
    print("ATENÇÃO: O segredo 'HF_TOKEN' não foi encontrado. Por favor, adicione seu token do Hugging Face aos segredos do Colab.")
    os.environ['HF_TOKEN'] = '' # Garante que a variável de ambiente seja definida, mesmo que vazia
except Exception as e:
    print(f"Ocorreu um erro ao carregar ou configurar o token do Hugging Face: {e}")
    os.environ['HF_TOKEN'] = '' # Garante que a variável de ambiente seja definida, mesmo que vazia

Token do Hugging Face carregado e configurado com sucesso via huggingface_hub.login().


In [ ]:
from datasets import load_dataset, concatenate_datasets
from tokenizers import ByteLevelBPETokenizer
from transformers import GPT2TokenizerFast
import os
import shutil

PASTA_PROJETO = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
pasta_tok = os.path.join(PASTA_PROJETO, "tokenizador")

# Verifica se o tokenizador já existe e o carrega, caso contrário, treina um novo.
if os.path.exists(pasta_tok) and os.path.isdir(pasta_tok) and \
   os.path.exists(os.path.join(pasta_tok, "vocab.json")) and \
   os.path.exists(os.path.join(pasta_tok, "merges.txt")):
    print(f"Tokenizador encontrado em: {pasta_tok}. Carregando tokenizador existente...")
    tokenizer = GPT2TokenizerFast.from_pretrained(pasta_tok, local_files_only=True)
    print("Tokenizador carregado com sucesso!")
else:
    print(f"Tokenizador não encontrado ou incompleto em {pasta_tok}. Treinando novo tokenizador...")

    # 1. Carrega frações exatas direto para o cache local do Colab (SEM streaming=True)
    # Dividido proporcionalmente para somar 50.000 artigos no total (50% EN, 25% PT, 25% ES)
    print("Baixando fatias da Wikipédia para a memória local (Isso leva cerca de 1-2 minutos)...")
    wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:25000]")
    wiki_pt = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train[:12500]")
    wiki_es = load_dataset("wikimedia/wikipedia", "20231101.es", split="train[:12500]")

    # Junta os datasets baixados localmente e embaralha na memória RAM
    print("Misturando e preparando os dados...")
    dataset_misto = concatenate_datasets([wiki_en, wiki_pt, wiki_es])
    dataset_misto = dataset_misto.shuffle(seed=42)

    # Gerador usado para alimentar o treinador do tokenizador direto da memória RAM
    def extrair_texto():
        for item in dataset_misto:
            yield item["text"]

    print("Treinando o tokenizador... Agora deve levar de 2 a 3 minutos.")
    tokenizer_raw = ByteLevelBPETokenizer()
    tokenizer_raw.train_from_iterator(
        extrair_texto(),
        vocab_size=50257, # Padrão clássico do GPT-2
        min_frequency=2,
        special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"]
    )

    # Salva o tokenizador no Drive
    # REMOVE o diretório existente antes de criar novamente para forçar a sincronização
    if os.path.exists(pasta_tok) and os.path.isdir(pasta_tok):
        print(f"Removendo diretório existente do tokenizador: {pasta_tok}")
        shutil.rmtree(pasta_tok)

    os.makedirs(pasta_tok, exist_ok=True) # Cria o diretório novamente
    tokenizer_raw.save_model(pasta_tok)

    # Converte para o formato utilizável pelo Hugging Face Trainer
    tokenizer = GPT2TokenizerFast.from_pretrained(pasta_tok, bos_token="<s>", eos_token="</s>", unk_token="<unk>", pad_token="<pad>", mask_token="<mask>")
    tokenizer.save_pretrained(pasta_tok)
    print(f"Tokenizador salvo com sucesso em: {pasta_tok}")


Tokenizador não encontrado ou incompleto em /content/drive/MyDrive/Colab_LLMs/Maia_Lite/tokenizador. Treinando novo tokenizador...
Baixando fatias da Wikipédia para a memória local (Isso leva cerca de 1-2 minutos)...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

20231101.en/train-00000-of-00041.parquet: reconstructing file:   0%|          |  0.00B /  420MB            

20231101.en/train-00000-of-00041.parquet: downloading bytes:           |  0.00B            

20231101.en/train-00001-of-00041.parquet: reconstructing file:   0%|          |  0.00B /  351MB            

...

20231101.en/train-00039-of-00041.parquet: reconstructing file:   0%|          |  0.00B /  465MB            

20231101.en/train-00039-of-00041.parquet: downloading bytes:           |  0.00B            

20231101.en/train-00040-of-00041.parquet: reconstructing file:   0%|          |  0.00B /  422MB            

20231101.en/train-00040-of-00041.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6407814 [00:00<?, ? examples/s]

In [ ]:
import os
import torch
from datasets import load_dataset, interleave_datasets
from transformers import GPT2Config, GPT2LMHeadModel, GPT2TokenizerFast, TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Adiciona esta linha para otimização de memória do PyTorch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

PASTA_PROJETO = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
pasta_tok = os.path.join(PASTA_PROJETO, "tokenizador")
pasta_saida = os.path.join(PASTA_PROJETO, "checkpoints_pretreino")

# VERIFICAÇÃO DE GPU
if torch.cuda.is_available():
    print(f"GPU disponível: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("ATENÇÃO: Nenhuma GPU disponível. O treinamento será executado na CPU, o que será muito mais lento.")
    device = torch.device("cpu")

# 1. Reload the tokenizer
tokenizer = GPT2TokenizerFast.from_pretrained(pasta_tok, local_files_only=True)

# 2. Configure the network to have exactly ~335 million parameters
config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=1024,   # Maximum supported context
    n_ctx=1024,
    n_embd=1024,        # Hidden dimension
    n_layer=24,         # 24 Transformer block layers
    n_head=16,          # 16 attention heads
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    use_cache=False,    # Necessário para gradient checkpointing
)
model = GPT2LMHeadModel(config)
model.to(device) # Move o modelo para a GPU ou CPU
print(f"Modelo inicializado com pesos aleatórios. Total de parâmetros: {model.num_parameters():,}")

# Ativa o gradient checkpointing para economizar memória
model.gradient_checkpointing_enable()

# 3. Load the datasets in streaming mode
wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
wiki_pt = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train", streaming=True)
wiki_es = load_dataset("wikimedia/wikipedia", "20231101.es", split="train", streaming=True)
dataset_misto = interleave_datasets([wiki_en, wiki_pt, wiki_es], probabilities=[0.5, 0.25, 0.25], seed=42)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=1024)

# Dynamic mapping because of streaming
tokenized_dataset = dataset_misto.map(tokenize_function, batched=True, remove_columns=["id", "url", "title", "text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 4. Training Parameters Optimized for the T4
training_args = TrainingArguments(
    output_dir=pasta_saida,
    max_steps=200000,
    per_device_train_batch_size=1, # Reduzido para 1 para economizar memória
    gradient_accumulation_steps=16, # Aumentado para 16 para manter o batch size efetivo de 16
    save_steps=2500,               # Save to Drive every ~1.5 hours of training
    save_total_limit=2,            # Keep the 2 latest checkpoints to save space
    logging_steps=500,
    fp16=True,                     # Mixed precision (essential na maioria das GPUs modernas)
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=2000,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# If the checkpoint directory contains data, resume automatically from where it stopped
checar_checkpoint = True if os.path.exists(pasta_saida) and len(os.listdir(pasta_saida)) > 0 else False

print(f"Iniciando treinamento. Continuando de checkpoint anterior? {checar_checkpoint}")
trainer.train(resume_from_checkpoint=checar_checkpoint)

# Save the consolidated final model
model.save_pretrained(os.path.join(PASTA_PROJETO, "modelo_335M_final"))
print("PRÉ-TREINO CONCLUÍDO COM SUCESSO!")
